# Exercise 1: Web Server Access Log Analysis

Student Name: Segundo
Student ID: TUPM-26-1952

Records Processed: 1025
Target Status Class: 4xx

In [3]:
import csv
import json
import shutil
from pathlib import Path
from collections import Counter

BASE_DIR = Path.cwd()

if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR / "Exercise1_Web_Server_Logs"

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

STUDENT_NAME = "Segundo"
STUDENT_ID = "TUPM-26-1952"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

print("Student:", STUDENT_NAME)
print("Student ID:", STUDENT_ID)
print("Records to process:", RECORDS_TO_PROCESS)
print("Target status class:", TARGET_STATUS_CLASS, "xx")

Student: Segundo
Student ID: TUPM-26-1952
Records to process: 1025
Target status class: 4 xx


In [4]:
input_file = DATA_DIR / "weblog.csv"

all_records = []
selected_records = []
status_counts = Counter()
url_counts = Counter()

with open(input_file, "r", encoding="utf-8-sig",
          newline="") as file:

    reader = csv.DictReader(file)

    for index, row in enumerate(reader):

        if index >= RECORDS_TO_PROCESS:
            break

        all_records.append(row)

        # Handle the original dataset's column spelling
        status = (
            row.get("Staus")
            or row.get("Status")
            or row.get("Response Status")
            or ""
        ).strip()

        try:
            status_code = int(status)
        except (ValueError, TypeError):
            continue

        status_class = status_code // 100

        status_counts[status_code] += 1

        if status_class == TARGET_STATUS_CLASS:
            selected_records.append(row)
            url = row.get("URL", "").strip()

            if url:
                url_counts[url] += 1

print("Records processed:", len(all_records))
print("Matching records:", len(selected_records))
print("Status counts:", dict(status_counts))

Records processed: 1025
Matching records: 5
Status counts: {200: 847, 302: 144, 304: 21, 206: 8, 404: 5}


In [5]:
print("\nREQUEST COUNT PER STATUS CODE")

for status, count in sorted(status_counts.items()):
    print(status, ":", count)

print("\nFIVE MOST REQUESTED URLS IN 4XX CLASS")

top_urls = url_counts.most_common(5)

for url, count in top_urls:
    print(url, ":", count)


REQUEST COUNT PER STATUS CODE
200 : 847
206 : 8
302 : 144
304 : 21
404 : 5

FIVE MOST REQUESTED URLS IN 4XX CLASS
GET /robots.txt HTTP/1.1 : 4
GET /favicon.ico HTTP/1.1 : 1


In [6]:
filtered_csv = OUTPUT_DIR / "filtered_web_logs.csv"

if selected_records:
    with open(filtered_csv, "w", encoding="utf-8",
              newline="") as file:

        writer = csv.DictWriter(
            file,
            fieldnames=selected_records[0].keys()
        )

        writer.writeheader()
        writer.writerows(selected_records)

    print("Filtered CSV created:", filtered_csv.name)
else:
    print("No matching records found.")

Filtered CSV created: filtered_web_logs.csv


In [7]:
summary = {
    "student_name": STUDENT_NAME,
    "student_id": STUDENT_ID,
    "records_processed": len(all_records),
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "matching_records": len(selected_records),
    "count_per_status_code": dict(status_counts),
    "five_most_requested_urls": [
        {"url": url, "count": count}
        for url, count in top_urls
    ]
}

json_file = OUTPUT_DIR / "web_log_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_file.name)

JSON summary created: web_log_summary.json


In [8]:
backup_file = BACKUP_DIR / "web_log_summary_backup.json"

shutil.copy2(json_file, backup_file)

if backup_file.exists():
    print("Backup verification: SUCCESS")
else:
    print("Backup verification: FAILED")

Backup verification: SUCCESS
